# tools

> The toolset is `shalya`. This module is what Ramabana adds to it, and the names it keeps resolving.

In [ ]:
#| default_exp tools

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from pathlib import Path
from fastcore.test import test_eq, test_fail
from ramabana.testing import FakeBackend, MemHost

def names(ts): return {t.__name__ for t in ts}

The host, the tool factories, the tool-result conventions and the skills registry moved to
[shalya](https://github.com/vedicreader/shalya). Ramabana re-exports every one of them, so
`from ramabana.tools import ...` keeps working and Leela's `sys.modules[__name__] = ramabana.tools`
keeps resolving.

What stays here needs an agent. Delegation needs a `Backend` and a `Run`. `draws_itself` needs a
`ModelSpec`. Neither is something a toolset should know about.

In [ ]:
#| export
import concurrent.futures, functools, json, re, threading, time, uuid
from fastcore.basics import AttrDict, ifnone
from fastcore.foundation import L
from fastcore.parallel import parallel
from shalya.core import (ERR, GIT_READ_TOOLS, GIT_TOOLS, GIT_WRITE_TOOLS, Hit, MAX_API, MAX_FILE,
                         MAX_GREP_HITS, MAX_HITS, MAX_TOOL_CHARS, apply_edits, clip,
                         clip_lines, cmds, diff_text, edits, err, failed, is_write, writes)
from shalya.host import (Capability, DENY, Host, HostError, LD_CHARS, LocalHost, MAX_VARS, NO_ROOTS,
                         SANDBOX, SECRET, SKIP_DIRS, SKIP_SUFFIXES, denied, implemented, ld_json, _fuse,
                         CodeHost, WebHost, NotebookHost, MemoryHost, WatchHost, SessionHost,
                         ShellHost, ApiHost, GitHost)
from shalya.skills import (EVENTS, EXTRA_MODULES, GROUP, MAX_SKILL_CHARS, Registry, SKILL_DESC_MAX,
                           Skill, discover, ext_dirs, find, load, skill_dirs, skill_index)
from shalya.tools import (API_VENDORS, GROUPS, IMAGE_API, IMAGE_MODEL, IMAGE_SIZES, RESPONSES_API,
                          _post_responses, api_model, api_tools, ask_tools, code_tools, file_tools,
                          image_available, media_dir, memory_tools, mime_for, notebook_tools,
                          readable, save_media, session_tools, shell_tools, skill_tools, watch_tools,
                          web_tools, git_tools)
from shalya.tools import image_tools as _image_tools
from shalya.tools import read_only as _read_only
from shalya.tools import tools_for as _tools_for
from ramabana.core import AgentError, agent_err, spec_caps
from ramabana.runtime import Run, current_run, run_context

In [ ]:
#| export
from fastcore.docments import frontmatter
from shalya.core import WRITE_TOOLS as _TOOL_WRITES

#: Names Leela imports from `ramabana.tools`. They are shalya's now, under the spellings Leela knows.
_cmds, _edits, _apply_edits, _diff = cmds, edits, apply_edits, diff_text

#: `cart_tools` is an extension rather than a host group, so its two spenders are added by name here.
WRITE_TOOLS = _TOOL_WRITES | {'cart_add', 'cart_remove'}

In [ ]:
#| export
#: shalya's names, re-exported so `from ramabana.tools import *` still finds them.
_all_ = ['frontmatter', 'API_VENDORS', 'Capability', 'DENY', 'ERR', 'EVENTS', 'EXTRA_MODULES', 'GIT_READ_TOOLS', 'GIT_TOOLS', 'GIT_WRITE_TOOLS', 'GROUP', 'GROUPS', 'Hit', 'Host', 'HostError', 'IMAGE_API', 'IMAGE_MODEL', 'IMAGE_SIZES', 'LD_CHARS', 'LocalHost', 'MAX_API', 'MAX_FILE', 'MAX_GREP_HITS', 'MAX_HITS', 'MAX_SKILL_CHARS', 'MAX_TOOL_CHARS', 'MAX_VARS', 'NO_ROOTS', 'RESPONSES_API', 'Registry', 'SANDBOX', 'SECRET', 'SKILL_DESC_MAX', 'SKIP_DIRS', 'SKIP_SUFFIXES', 'Skill', '_apply_edits', '_cmds', '_diff', '_edits', 'api_model', 'api_tools', 'ask_tools', 'apply_edits', 'clip', 'clip_lines', 'cmds', 'code_tools', 'denied', 'diff_text', 'discover', 'edits', 'err', 'ext_dirs', 'failed', 'file_tools', 'find', 'git_tools', 'image_available', 'implemented', 'is_write', 'ld_json', '_fuse', 'CodeHost', 'WebHost', 'NotebookHost', 'MemoryHost', 'WatchHost', 'SessionHost', 'ShellHost', 'ApiHost', 'GitHost', 'load', 'media_dir', 'memory_tools', 'mime_for', 'notebook_tools', 'readable', 'save_media', 'session_tools', 'shell_tools', 'skill_dirs', 'skill_index', 'skill_tools', 'watch_tools', 'web_tools', 'writes']

## A host with nothing behind it

`NullHost` is what the harness runs on in a test: the path boundary, refusing. It declares no
capability group beyond the file one every host has, so `tools_for` gives it the file tools and
nothing else.

In [ ]:
#| export
@implemented
class NullHost(Host):
    "A host with nothing behind it. The reference implementation of 'this host cannot'."

    def __init__(self, roots=()): self._roots = [str(r) for r in roots]

    @property
    def roots(self): return self._roots
    def check(self, path, must_exist=False, reading=False): raise HostError(f'{NO_ROOTS}: {path}')
    def walk(self): return []
    def read(self, path): return None
    def write(self, path, text): raise HostError('this host cannot write')
    def text_at(self, path): return None

## Drawing

Whether the turn's own model can draw is a fact about the model, so shalya takes it as a callable and
this is where the callable comes from. `Caps.tools` is the answer that matters:
`supported_output_modalities` says no for every chat model, because it describes what one returns
unprompted.

In [ ]:
#| export
def draws_itself(spec):
    "Can `spec`'s own model draw, given the image tool?"
    if spec is None: return False
    c = spec_caps(spec)
    return bool(c is not None and 'image' in getattr(c, 'tools', ()))

def _from_responses(raw):
    "Generated pictures out of a Responses reply, read by the same `rishi` code a turn uses."
    from rishi.remote import gen_media
    return gen_media(raw)

def image_tools(host, mx=MAX_TOOL_CHARS, session='', get_spec=None, on_media=None):
    "shalya's image group, told what this turn's model can do."
    spec = get_spec() if get_spec else None
    return _image_tools(host, mx, session, draws_itself=lambda: draws_itself(spec),
                        from_reply=_from_responses, model_id=getattr(spec, 'model_id', ''),
                        on_media=on_media)

## Assembling the tool list

`tools_for` is shalya's. Ramabana keeps its own signature, because `get_spec` and `on_media` belong
to a turn rather than to a host, and builds the image group before handing it over.

In [ ]:
#| export
def tools_for(host, get_skills=None, extra=(), mx=MAX_TOOL_CHARS, drop=(), get_spec=None, on_media=None):
    """Every tool this host declares it can support, plus whatever extensions registered.

    Groups come from `Host.provides`. `drop` withholds a group the host does support, decided by
    `core.budget_for` and reported by `Agent.budget`.
    """
    # credentialled, never probed: a key is either there or it is not
    image = (image_tools(host, mx, get_spec=get_spec, on_media=on_media)
             if image_available() and 'image' not in set(drop or ()) else None)
    return _tools_for(host, get_skills=get_skills, extra=extra, mx=mx, drop=drop, image=image)

In [ ]:
h = NullHost(['/proj'])
sorted(h.provides), [t.__name__ for t in tools_for(h)]

In [ ]:
test_eq(sorted(h.provides), ['file'])
test_eq(h.roots, ['/proj'])
test_fail(lambda: h.check('a.py'), contains='no folders are open')

A real host over a real folder, and the tools it earns. `ts` is what the sub-agent section filters
below.

In [ ]:
root = Path(tempfile.mkdtemp()).resolve()/'proj'
(root/'pkg').mkdir(parents=True)
(root/'pkg'/'sizes.py').write_text('def threshold(n):\n    "Half of n."\n    return n // 2\n')
local = LocalHost([root], index=False)
ts = tools_for(local)
sorted(local.provides), len(ts)

In [ ]:
assert {'search_code', 'view_file', 'run_python', 'run_shell'} <= names(ts), sorted(names(ts))
assert not (names(ts) & {'memory_tree', 'api_load', 'ask_memory'})   # no store, no specs
test_eq(local.can('file'), True)

## Sub-agents

Delegation is a context strategy, not a speed one. A broad question that takes twenty tool calls to answer costs the caller one question and one answer, because the sub-agent's working is discarded with its conversation. A sub-agent gets read-only tools, and cannot delegate further: recursion here is a fan-out tree whose width nobody chose.

In [ ]:
#| export
SUB_MAX_STEPS = 12

SUB_SP = """You are a research sub-agent inside a Python IDE. Another agent has delegated one \
question to you because answering it takes many tool calls and the answer is short.

- Answer exactly the question asked. Nothing else.
- Use your tools as much as you need; nobody is paying attention to how many calls it takes.
- Report what you found, with file paths and line numbers, not what you infer or expect.
- If the answer is that there is nothing, say so plainly. A confident wrong answer is far \
worse than "no matches, and here is what I searched for".
- You cannot edit anything. If the answer implies a change, describe the change and stop.
- `inspect_python` answers questions about the user's live variables without changing them. \
Its default scope is a sandbox that refuses most library calls; pass `scope='overlay'` to \
get the real interpreter. Use it rather than guessing at what is in memory."""


#: Swapped in for the two read-only lines above when a session grants sub-agents writes.
SUB_WRITE_SP = """- You have the delegating agent's write tools as well as its read tools: create and \
edit files, run commands, run Python. Every call is recorded on the session and goes through the \
approval policy the main agent answers to. A refusal comes back with a reason. Read it and change \
the approach.
- Write only what the task asked for. You cannot see the conversation that sent you. Anything \
else you change is a change nobody reviewed.
- Verify with the tool that proves it. Run the test. Read the file back. Report the evidence.
- `run_python` shares the user's kernel namespace. Bind results to NEW names. You cannot rebind or \
delete what the user made."""


def sub_briefing(writes=False):
    "The sub-agent standing instructions, with the read-only sentences swapped out when writes are on."
    if not writes: return SUB_SP
    keep = [ln for ln in SUB_SP.splitlines()
            if not ln.startswith('- You cannot edit anything') and not ln.startswith('- `inspect_python`')]
    return '\n'.join(keep).rstrip() + '\n' + SUB_WRITE_SP


# A sub-agent does not spawn sub-agents: recursion here is a fan-out tree whose width nobody chose.
# Nor does it open standing work: a folder watch outlives the task that opened it, and nobody
# asked for the reviews it would keep producing after the delegation is forgotten.
NO_SUB = frozenset({'delegate_search', 'delegate_parallel',
                    'watch_folder', 'cancel_folder_watch', 'check_folders'})

In [ ]:
#| export
def read_only(tools, max_calls=None, writes=False):
    "The tools a sub-agent may have, optionally behind a hard per-task call budget."
    blocked = NO_SUB if writes else (WRITE_TOOLS | NO_SUB)
    allowed = [t for t in tools if getattr(t, '__name__', '') not in blocked]
    if max_calls is None: return allowed
    state, lock = {'n': 0}, threading.Lock()

    def guarded(f):
        @functools.wraps(f)
        def call(*args, **kw):
            with lock:
                state['n'] += 1
                over = state['n'] > max_calls
            if over:
                return ('Sub-agent tool budget exhausted. Stop calling tools and return the '
                        'best evidence-backed answer now.')
            return f(*args, **kw)
        return call
    return [guarded(t) for t in allowed]

Every write tool and both delegation tools are filtered out, whatever else the host offered:

In [ ]:
[t.__name__ for t in read_only(ts)]

['search_code',
 'grep',
 'ls',
 'similar_code',
 'outline',
 'list_files',
 'view_file',
 'web_search',
 'read_url',
 'research']

In [ ]:
test_eq(set(t.__name__ for t in read_only(ts)) & WRITE_TOOLS, set())
sorted(NO_SUB)

['delegate_parallel', 'delegate_search']

With a budget, the tools themselves stop the loop. Local engines own their internal tool loop. The wrapper is the one hard stop that works on every backend.

In [ ]:
budgeted = read_only(ts, max_calls=1)
search = next(t for t in budgeted if t.__name__ == 'search_code')
search('return'), search('return')

('[memory]\n/proj/a.py:1    def a(): return 1\n  FILE -- use this exact path with view_file/edit_file\n/proj/b.py:1    def b(): return a() + 1\n  FILE -- use this exact path with view_file/edit_file',
 'Sub-agent tool budget exhausted. Stop calling tools and return the best evidence-backed answer now.')

`delegate` runs one question in a throwaway conversation on the same engine, and closes it in a `finally`. A sub-agent whose context leaks back into the session is just a slower way of doing the work inline.

In [ ]:
#| export
def sub_sp(sp=SUB_SP, skills=()):
    "A sub-agent's briefing: its standing instructions, then the bodies of the skills its task named."
    if not skills: return sp
    return sp + '\n\n' + '\n\n'.join(f'## {s.name}\n\n{s.text()}' for s in skills)


def _stopped(run):
    "A stopped delegation answers in text: a dict would reach the model as its own repr."
    return f'The delegated question was stopped ({run.state}) before it answered.'


def bad_json(e, span=120):
    "The fragment a JSON error is about."
    doc, pos = getattr(e, 'doc', None), getattr(e, 'pos', None)
    if not isinstance(doc, str) or not isinstance(pos, int): return ''
    lead = '…' if pos > span else ''
    tail = '…' if pos + span < len(doc) else ''
    return f'\nit stopped here: {lead}{doc[max(0, pos - span):pos + span]}{tail}'


def _model_refused(sub, reply):
    "Whether what came back is the sub-agent's backend reporting its own failure, not an answer."
    problems = getattr(sub, 'problems', None) or []
    return bool(problems) and str(reply or '').strip() == str(problems[-1]).strip()


def delegate(backend, question, tools=(), sp=None, max_steps=SUB_MAX_STEPS, skills=(),
             writes=False,      # hand over WRITE_TOOLS as well
             approve=None,      # the gate those writes answer to, which `spawn` inherits none of
             run=None):         # a pre-registered child run
    "Ask `question` in a throwaway conversation on `backend`'s engine. Returns the answer text."
    sub = None
    run = run or Run(f'run_{uuid.uuid4().hex[:12]}', 'child', str(question), backend.spec.name, current_run())
    if not run.start(): return _stopped(run)
    try:
        # the tool wrappers are the hard stop: native engines own their own tool loop
        kw = {'approve': approve} if approve is not None else {}
        sub = backend.spawn(sp=sub_sp(ifnone(sp, sub_briefing(writes)), skills),
                            tools=read_only(tools, max_calls=max_steps * 4, writes=writes), **kw)
        if hasattr(sub, 'max_steps'): sub.max_steps = max_steps
        if not run.attach(sub): return _stopped(run)
        with run_context(run): reply = sub.send(question, run=run)
        if run.cancelled: return _stopped(run.finish())
        if _model_refused(sub, reply):
            run.finish('failed')
            return err(f'delegation failed after the sub-agent was asked: {reply}')
        run.finish()
        return _delegate_result(reply)
    except Exception as e:
        run.finish('failed')
        return err('delegation failed after the sub-agent was asked' if sub is not None else
                   'delegation failed before the sub-agent was asked, so nothing was sent', e) + bad_json(e)
    finally:
        if sub is not None:
            try: sub.close()
            except Exception: pass

In [ ]:
#| export
def delegate_many(backend, questions, tools=(), sp=None, max_steps=SUB_MAX_STEPS, n_workers=4,
                  skills=(), writes=False, approve=None, parent=None):
    "Ask several questions. Register every child before starting serial or parallel workers."
    qs = L(questions)
    if not qs: return L()
    parent = parent or current_run()
    runs = [Run(f'run_{uuid.uuid4().hex[:12]}', 'child', str(q), backend.spec.name, parent,
                getattr(parent, 'grace', .25)) for q in qs]
    def run(item):
        q, child = item
        if child.cancelled:return _stopped(child)
        return delegate(backend, q, tools, sp, max_steps, skills, writes, approve, child)
    items = list(zip(qs, runs))
    # writing sub-agents stay serial. `Approvals` holds one pending ask, and nobody can review
    # concurrent edits to one workspace
    workers = 1 if writes or getattr(backend.spec, 'local', False) else min(n_workers, len(qs))
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=max(1, workers))
    futures = [ex.submit(run, item) for item in items]
    try:
        while True:
            if all(f.done() for f in futures):break
            if parent is not None and parent.cancelled:
                concurrent.futures.wait(futures, timeout=getattr(parent, 'grace', .25))
                break
            time.sleep(.005)
        out = []
        for child, future in zip(runs, futures):
            if future.done():
                try:out.append(future.result())
                except Exception as e:out.append(err('delegation failed', e) + bad_json(e))
            else:
                child.detach(); out.append(_stopped(child))
        return L(out)
    finally:ex.shutdown(wait=False, cancel_futures=True)

In [ ]:
be = FakeBackend(replies=['the caller never sees this'])
delegate(be, 'which files import fastllm?', tools=ts)

'sub answer'

The spawned conversation is separate, and gone by the time the answer is returned.

In [ ]:
test_eq(len(be.spawned), 1)
be.spawned[0].hist

[{'role': 'user', 'content': 'which files import fastllm?'},
 {'role': 'assistant', 'content': 'sub answer'}]

`delegate_many` keeps the answers in the order the questions were asked. On a local model it runs them one after another on purpose: litert holds one conversation at a time. Fanning out would mean racing for the same engine to find out what happens.

In [ ]:
delegate_many(be, ['what imports fastllm?', 'where is compaction triggered?'], tools=ts)

['sub answer', 'sub answer']

The tools themselves take callables rather than a backend. A model switch mid-session is picked up. The tool the model is holding must not be pinned to whichever backend happened to be current when the list was built.

In [ ]:
#| export
def named_skills(get_skills, names):
    "The skills a delegated task named, and a note about any name that matched nothing."
    if not names or get_skills is None: return [], ''
    every = list(get_skills() or [])
    got, missing = [], []
    for n in [x for x in str(names).replace(',', ' ').split() if x]:
        s = find(every, n)
        got.append(s) if s is not None else missing.append(n)
    if not missing: return got, ''
    return got, (f"\n\n[no skill named {', '.join(missing)}; this repository has "
                 f"{', '.join(s.name for s in every) or 'none'}]")


def subagent_tools(get_backend, get_tools, get_skills=None, get_cloud_backend=None,
                   get_writes=None,     # the session's sub-agent write toggle, read per call
                   get_approve=None):   # the gate those writes answer to
    """The `delegate` tool, bound to whatever backend routing says sub-agents run on.

    Every argument is a callable. A model switch mid-session is picked up. `get_tools` is the
    sub-agent model's tool list, not the turn model's.
    """

    def _writes(): return bool(get_writes()) if get_writes is not None else False
    def _approve(): return get_approve() if (get_approve is not None and _writes()) else None

    def delegate_search(question: str, skills: str = '') -> str:
        """Hand a broad search question to a sub-agent and get back only its conclusion.

        Use this when answering would take many `search_code` / `view_file` / `read_url` /
        `inspect_python` calls whose results you do not need to keep. "where else do we
        do X", "which files import Y", "what shape is everything in this namespace". Its
        working is discarded. The cost to your context is one question and one answer.

        The sub-agent has your read-only tools. Whether it also has your write tools is the
        session's setting rather than yours. With sub-agent writes on it can edit, run commands
        and run Python under the approval policy you answer to. The task you send may then ask
        for a change. With them off it can only report.

        `skills` names skills from your skill index, comma separated, whose text the sub-agent
        should start with: name the one or two its task actually needs. You hold the index and
        it does not. This is the only way it gets a skill without spending a step reading
        one. Leave it empty when the task needs no particular skill.

        Ask one self-contained question. The sub-agent cannot see this conversation.
        """
        b = get_backend()
        if b is None: return 'no model is available to delegate to'
        sk, note = named_skills(get_skills, skills)
        return clip(delegate(b, question, get_tools(), skills=sk, writes=_writes(),
                             approve=_approve()), MAX_TOOL_CHARS) + note

    def delegate_parallel(questions: str, skills: str = '', cloud_model: str = '') -> str:
        """Hand several independent questions to sub-agents at once, and get back every answer.

        `questions` is a JSON array of strings, e.g.
          ["which files import fastllm?", "where is compaction triggered?", "what is df's shape?"]

        Use it when you have two or more questions that do not depend on each other. They
        run concurrently, each in its own throwaway conversation with your read-only tools. Three questions cost you three short answers rather than the sixty tool results
        it would take to answer them yourself. With sub-agent writes on they run one after
        another instead, because their approvals share one queue.

        `skills` names skills from your skill index, comma separated, given to every one of
        them. Use it when the questions share a subject. When they do not, ask them in separate
        `delegate_search` calls so each gets only what its own task needs.

        `cloud_model` optionally selects one configured remote model for this fan-out. It does not
        change the session's turn or default sub-agent model. Every question must be self-contained:
        a sub-agent cannot see this conversation or the other questions.
        """
        b = get_cloud_backend(cloud_model) if cloud_model and get_cloud_backend is not None else get_backend()
        if b is None: return f"no model is available to delegate to{f' ({cloud_model})' if cloud_model else ''}"
        try:
            qs = json.loads(questions) if isinstance(questions, str) else list(questions)
            if not isinstance(qs, list) or not all(isinstance(q, str) for q in qs):
                raise ValueError('expected a JSON array of strings')
        except Exception as e:
            return err('could not parse questions', e)
        if not qs: return 'no questions given'
        sk, note = named_skills(get_skills, skills)
        answers = delegate_many(b, qs, get_tools(), skills=sk, writes=_writes(), approve=_approve())
        return clip('\n\n'.join(f'### {q}\n{a}' for q, a in zip(qs, answers)), MAX_TOOL_CHARS * 2) + note

    return [delegate_search, delegate_parallel]

In [ ]:
delegate_search, delegate_parallel = subagent_tools(lambda: be, lambda: ts)
[t.__name__ for t in subagent_tools(lambda: be, lambda: ts)]

['delegate_search', 'delegate_parallel']

With no model available it says so, rather than raising into the turn.

In [ ]:
test_eq(subagent_tools(lambda: None, lambda: ts)[0]('anything'), 'no model is available to delegate to')
delegate_parallel('["what imports fastllm?"]')

'### what imports fastllm?\nsub answer'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| export
def _delegate_result(text):
    "Reject degenerate delegated prose before it can be presented as research."
    text = str(text or '').strip()
    words = re.findall(r"[A-Za-z0-9_+.-]+", text.lower())
    if not text: return 'Delegated inspection failed: the sub-agent returned no answer.'
    if len(words) >= 12 and max(words.count(w) for w in set(words)) > max(8, len(words) // 4):
        return 'Delegated inspection failed: repetitive output was discarded as unreliable.'
    return text

Parallel delegation registers every child before work starts. Cancelling the parent stops the running child and prevents queued children from spawning.


In [ ]:
class _Sub:
    max_steps = 0
    def __init__(self, owner): self.owner, self.release = owner, threading.Event()
    def send(self, question, run=None):
        self.owner.started.append(question); self.release.wait(); return 'late'
    def cancel(self): self.owner.cancelled += 1; self.release.set(); return True
    def close(self): pass

class _ParentBackend:
    def __init__(self):
        self.spec = AttrDict(name='fake-child', local=False)
        self.started, self.spawned, self.cancelled = [], 0, 0
    def spawn(self, **kw): self.spawned += 1; return _Sub(self)

backend, parent, box = _ParentBackend(), Run('run_parent', grace=.03), []
parent.start()
t = threading.Thread(target=lambda: box.extend(delegate_many(backend, ['a', 'b', 'c'], n_workers=1, parent=parent)), daemon=True)
t.start()
while not backend.started: time.sleep(.001)
test_eq(len(parent.children), 3)
parent.cancel(); t.join(.2)
test_eq((backend.spawned, backend.cancelled), (1, 1))
test_eq(len(box), 3)
# either shape says cancelled, and which one comes back is a matter of whether the worker finished
# inside the join: a future still in flight is detached and reported as its run, one that returned
# carries `delegate`'s own cancellation message. Asserting only the first made this a coin flip
assert all((isinstance(x, dict) and x['state'] in ('cancelled', 'detached'))
           or (isinstance(x, str) and 'cancelled' in x) for x in box)
